# INbreast preprocessing

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

## Load raw data

In [ ]:
from preprocessing.inbreast import Inbreast, csv_save_path
from utils.preprocessing import birads_assessment, birads_mapping, breast_density, dview, get_value, laterality, yes_no_mapping

ds = Inbreast()
ds.set_dataset_name("inbreast")
ds.clinical_df.head()

### Step 1 — normalize file name, breast density (ACR), and annotation status

In [ ]:
ds.clinical_df["file name"] = ds.clinical_df["file name"].apply(lambda x: str(int(x)))
ds.clinical_df["acr"] = ds.clinical_df["acr"].apply(lambda x: get_value(x, breast_density))
ds.clinical_df["lesion annotation status"] = ds.clinical_df["lesion annotation status"].apply(
    lambda x: str(x).strip().lower() if pd.notna(x) else None
)
ds.clinical_df[["acr", "lesion annotation status"]].apply(lambda c: c.value_counts(dropna=False))

### Step 2 — map laterality and view, then drop the single 'from below' exam

In [ ]:
ds.clinical_df["laterality"] = ds.clinical_df["laterality"].apply(lambda x: get_value(x, laterality))
ds.clinical_df["view"] = ds.clinical_df["view"].apply(lambda x: get_value(x, dview))
ds.clinical_df = ds.clinical_df[ds.clinical_df["view"] != "from below"].reset_index(drop=True)
ds.clinical_df["view"].value_counts(dropna=False)

### Step 3 — map BI-RADS and the yes/no finding flags (mass, micros, distortion, asymmetry)

In [ ]:
ds.clinical_df["bi-rads"] = ds.clinical_df["bi-rads"].apply(lambda x: get_value(x, birads_assessment))
ds.clinical_df["mass"] = ds.clinical_df["mass"].apply(lambda x: get_value(x, yes_no_mapping) if pd.notna(x) else None)
ds.clinical_df["micros"] = ds.clinical_df["micros"].apply(lambda x: get_value(x, yes_no_mapping) if pd.notna(x) else None)
ds.clinical_df["distortion"] = ds.clinical_df["distortion"].apply(
    lambda x: get_value(x, yes_no_mapping) if pd.notna(x) and len(str(x).strip().lower()) > 0 else None
)
ds.clinical_df["asymmetry"] = ds.clinical_df["asymmetry"].apply(lambda x: get_value(x, yes_no_mapping) if pd.notna(x) else None)
ds.clinical_df["pectoral muscle annotation"] = ds.clinical_df["pectoral muscle annotation"].apply(
    lambda x: str(x).strip().lower() if pd.notna(x) and len(str(x).strip().lower()) > 0 else None
)
ds.clinical_df["bi-rads"].value_counts(dropna=False)

### Step 4 — fold known-biopsy-proven (6) into highly-suggestive (5), then default missing finding flags to 'no'

In [ ]:
ds.clinical_df["bi-rads"] = ds.clinical_df["bi-rads"].replace({birads_mapping[6]: birads_mapping[5]})
ds.clinical_df.fillna({"mass": "no", "micros": "no", "distortion": "no", "asymmetry": "no"}, inplace=True)
ds.clinical_df["bi-rads"].value_counts(dropna=False)

### Step 5 — keep only exclusive mass-XOR-microcalcification cases with no distortion/asymmetry

In [ ]:
ds.clinical_df = ds.clinical_df[
    ((ds.clinical_df["mass"] == "yes") ^ (ds.clinical_df["micros"] == "yes"))
    & (ds.clinical_df["distortion"] == "no")
    & (ds.clinical_df["asymmetry"] == "no")
].reset_index(drop=True)
ds.clinical_df.shape

### Step 6 — rename columns for the common schema

In [ ]:
ds.clinical_df.rename(columns={"acr": "breast density", "micros": "has microcalcifications", "view": "exam view"}, inplace=True)
ds.clinical_df.columns.tolist()

## Build one exam record

In [ ]:
row = ds.clinical_df.iloc[0]
exam = ds.process_row(row)
exam

## Final processed CSV

In [ ]:
import json

final_df = pd.read_csv(csv_save_path)
print(f"rows before per-exam processing: {len(ds.clinical_df)} | rows in final csv: {len(final_df)}")
final_df.head()

In [ ]:
sample = final_df.iloc[0]
print(json.dumps(json.loads(sample['context']), indent=2))
print(json.dumps(json.loads(sample['findings']), indent=2))